In [1]:
import json
from pathlib import Path

import pandas as pd
from boulderopalscaleupplots.plots import Plotter
from boulderopalscaleupsdk.plotting.dtypes import LinePlot
from scaleupcore.analysis.dtypes import AnalysisResult
from scaleupdevtools.spin.equal1_tools.replay_db_tools import replay_record


2026-06-04 13:43:15,797 - qm - INFO     - Starting session: 699fa51b-6bd4-41c7-b264-36befd04efea


In [5]:
job_file = Path.home() / (
    "Documents/repositories/boulder-opal-scale-up/examples/scaleup/equal1_calibration_store/calibration_data/"
    "one_qubit_rb-2026-05-16T04:28:55.031094+00:00.json"
)

with job_file.open() as f:
    job_data = json.load(f)

job_id = job_data["id"]
entry = job_data["calibration_log"][0]

# Override min_valid_shots here — original value comes from the stored parameters
params = {**entry["parameters"], "min_valid_shots": 100}  # <-- adjust as needed

record = {
    "job_id": job_id,
    "experiment_name": entry["experiment_name"],
    "parameters": params,
    "measurements": entry["measurements"],
}

print(f"Replaying job: {job_id}")  # noqa: T201
replay = replay_record(record)

# ── Plot (averaged + fit only) ────────────────────────────────────────────────
for plot in replay.plots:
    if isinstance(plot, LinePlot):
        plot = plot.model_copy(
            update={"lines": [l for l in plot.lines if l.label != "Individual Sequences"]}
        )
    Plotter(plot, dark_mode=True).figure.show()

# ── Export to CSV ─────────────────────────────────────────────────────────────
csv_path = Path.home() / "Downloads" / f"{job_id.replace(':', '-')}_rb_data.csv"  # <-- adjust path

analysis_value = replay.action.updates.value
if isinstance(analysis_value, AnalysisResult):
    fit_result = analysis_value.value

    averaged_df = pd.DataFrame({
        "type": "averaged",
        "clifford_depth": params["clifford_depths"],
        "p_ground": fit_result.probability_ground,
        "p_ground_std": fit_result.classified_std,
    })

    fit_df = pd.DataFrame({
        "type": "fit",
        "clifford_depth": fit_result.x_fit,
        "p_ground": fit_result.y_fit,
        "p_ground_std": float("nan"),
    })

    pd.concat([averaged_df, fit_df], ignore_index=True).to_csv(csv_path, index=False)
    print(f"Saved to {csv_path}")  # noqa: T201
else:
    print(f"Analysis failed — no CSV written: {analysis_value.message}")  # noqa: T201


Replaying job: one_qubit_rb-2026-05-16T04:28:55.031094+00:00


/Users/jpdodson/Documents/repositories/boulder-opal-scale-up/packages/scale-up-spin/scaleupspin/experiments/one_qubit_rb/analysis.py:358: RuntimeWarning: Mean of empty slice
  probability_ground_individual = np.nanmean(


Saved to /Users/jpdodson/Downloads/one_qubit_rb-2026-05-16T04-28-55.031094+00-00_rb_data.csv


In [12]:
import re

import plotly.colors
import plotly.graph_objects as go

data_path = Path.home() / "Documents/repositories/boulder-opal-scale-up/examples/scaleup/equal1_calibration_store/calibration_data/"

job_files = [
    data_path / "one_qubit_rb-2026-05-16T04:28:55.031094+00:00.json",
    data_path / "one_qubit_rb-2026-05-16T02:36:01.630621+00:00.json",
]

colors = plotly.colors.qualitative.Plotly
fig = go.Figure()

for i, job_file in enumerate(job_files):
    color = colors[i % len(colors)]

    with job_file.open() as f:
        job_data = json.load(f)

    job_id = job_data["id"]
    entry = job_data["calibration_log"][0]
    params = {**entry["parameters"], "min_valid_shots": 100}

    record = {
        "job_id": job_id,
        "experiment_name": entry["experiment_name"],
        "parameters": params,
        "measurements": entry["measurements"],
    }

    replay = replay_record(record)
    analysis_value = replay.action.updates.value

    if isinstance(analysis_value, AnalysisResult):
        fit_result = analysis_value.value

        # Qubit name is in subtitle: "qubit dot0, X gate"
        qubit_name = job_file.stem  # fallback
        for plot in replay.plots:
            if isinstance(plot, LinePlot) and plot.config.subtitle:
                match = re.search(r"qubit\s+(\S+?)(?:,|\s|$)", plot.config.subtitle)
                if match:
                    qubit_name = match.group(1)
                break

        fidelity_pct = fit_result.fidelity.value * 100
        label = f"{qubit_name} ({fidelity_pct:.2f}%)"

        fig.add_trace(go.Scatter(
            x=params["clifford_depths"],
            y=fit_result.probability_ground,
            mode="markers",
            name=label,
            marker=dict(color=color),
            legendgroup=label,
        ))
        fig.add_trace(go.Scatter(
            x=fit_result.x_fit,
            y=fit_result.y_fit,
            mode="lines",
            name=label,
            line=dict(color=color),
            legendgroup=label,
            showlegend=False,
        ))
    else:
        print(f"Skipping {job_file.name}: {analysis_value.message}")  # noqa: T201

fig.update_layout(
    template="plotly_dark",
    xaxis=dict(title="Clifford Depth", type="log"),
    yaxis_title="P(0)",
    title="One-Qubit RB",
)
fig.show()


/Users/jpdodson/Documents/repositories/boulder-opal-scale-up/packages/scale-up-spin/scaleupspin/experiments/one_qubit_rb/analysis.py:358: RuntimeWarning:

Mean of empty slice

/Users/jpdodson/Documents/repositories/boulder-opal-scale-up/packages/scale-up-spin/scaleupspin/experiments/one_qubit_rb/analysis.py:358: RuntimeWarning:

Mean of empty slice

